# Experiment

## Import libraries

In [25]:
import pandas as pd

file_path = "treg_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)


# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="month_str",
    value_name="value",
)

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Extract year and month
df["date"] = pd.to_datetime(df["month_str"], errors="coerce")

# Drop rows where date couldn't be parsed
df = df.dropna(subset=["date"])

# Extract numeric year, month
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

# Use pivot_table with first() to handle duplicates
df = df.pivot_table(
    index=["year", "month"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by year and month
df = df.sort_values(["year", "month"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21980\1733260966.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["month_str"], errors="coerce")


Chỉ tiêu,year,month,international_liquidity_total_reserves_excluding_gold_foreign_exchange_us_dollars,international_liquidity_total_reserves_excluding_gold_us_dollars
0,2000,3,3589.29,3591.62
1,2000,6,3531.38,3531.89
2,2000,9,3537.72,3541.17
3,2000,12,3416.18,3416.51
4,2001,1,3435.00,3445.07
...,...,...,...,...
292,2025,1,78985.97,80853.01
293,2025,2,79471.09,81345.68
294,2025,3,80650.31,82553.10
295,2025,4,80096.30,82038.34


In [26]:
df.columns

Index(['year', 'month',
       'international_liquidity_total_reserves_excluding_gold_foreign_exchange_us_dollars',
       'international_liquidity_total_reserves_excluding_gold_us_dollars'],
      dtype='object', name='Chỉ tiêu')